In [1]:
pip install -U langchain langchain-community langchain-core langchain-ollama langchain-chroma langchain-text-splitters beautifulsoup4


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## MultiQuery

In [8]:
import os
from getpass import getpass

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = getpass("Enter LangSmith API key: ")
os.environ["LANGSMITH_PROJECT"] = "rag-from-scratch"

In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2", temperature=0)

response = llm.invoke("What is RAG?")
print(response.content)

RAG can refer to different things, but some common meanings include:

1. RAG (Radio Active Gas): In nuclear physics, RAG refers to radioactive gases, such as radon or xenon.
2. RAG (Resin-Assisted Grinding): In materials science and manufacturing, RAG is a process used to create complex shapes and structures by combining resin with abrasive particles.
3. RAG (Racial And Geographic) disparities: In sociology and social justice, RAG refers to the study of disparities in health, education, and economic outcomes that are influenced by both racial and geographic factors.
4. RAG (Rapid Action Group): In some organizations, RAG is an acronym for a rapid response team or a group that responds quickly to emergencies or crises.
5. RAG (Racing And Gaming): In the gaming and racing communities, RAG can refer to a racing or gaming team or organization.

Without more context, it's difficult to determine which definition is most relevant. If you have any additional information or clarification, I'd b

In [6]:
#### INDEXING ####

# Load blog
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

blog_docs = loader.load()

# Split documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)

splits = text_splitter.split_documents(blog_docs)

# Create embeddings and vector database
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

print("Documents loaded:", len(blog_docs))
print("Chunks created:", len(splits))
print("Vector database created successfully!")

/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_14782/2444823316.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Documents loaded: 1
Chunks created: 50
Vector database created successfully!


In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines. Original question: {question}"""

prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_perspectives
    | llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [7]:
from langchain_core.load import dumps, loads

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    flattened_docs = [
        dumps(doc) for sublist in documents for doc in sublist
    ]
    unique_docs = list(set(flattened_docs))
    return [loads(doc) for doc in unique_docs]

# Retrieve
question = "What is task decomposition for LLM agents?"

retrieval_chain = (
    generate_queries
    | retriever.map()
    | get_unique_union
)

docs = retrieval_chain.invoke({"question": question})
len(docs)

/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_11110/3409524559.py:9: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_11110/3409524559.py:9: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


13

In [8]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# Using the Ollama model initialized earlier
# llm = ChatOllama(model="llama3.2", temperature=0)

final_rag_chain = (
    {"context": retrieval_chain,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question": question})

/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_11110/3409524559.py:9: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


'Task decomposition for LLM (Large Language Model) agents involves breaking down large tasks into smaller, manageable subgoals. This allows the agent to efficiently handle complex tasks. There are three methods of task decomposition:\n\n1. Simple prompting: The LLM is instructed to "think step by step" to decompose hard tasks into smaller and simpler steps.\n2. Task-specific instructions: The LLM is given specific instructions, such as "Write a story outline."\n3. Human inputs: Human inputs are used to guide the task decomposition process.\n\nTask decomposition can be done using a combination of these methods, and the choice of method depends on the specific task and the capabilities of the LLM agent.'

In [9]:
# Rag-Fusion

## Rag Fusion

In [10]:
from langchain_core.prompts import ChatPromptTemplate

# RAG-Fusion: Related
template = """You are a helpful assistant that generates multiple search queries based on a single input query.

Generate multiple search queries related to: {question}

Output (4 queries):"""

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [11]:
from langchain_core.output_parsers import StrOutputParser

generate_queries = (
    prompt_rag_fusion
    | llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [12]:
from langchain_core.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents
        and an optional parameter k used in the RRF formula """

    # Initialize a dictionary to hold fused scores
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document with its rank
        for rank, doc in enumerate(docs):

            # Convert the document to a string
            doc_str = dumps(doc)

            # Initialize its score if not already present
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0

            # Update the score using the RRF formula
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort documents based on fused scores
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(
            fused_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]

    return reranked_results


retrieval_chain_rag_fusion = (
    generate_queries
    | retriever.map()
    | reciprocal_rank_fusion
)

docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_11110/753188772.py:28: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


12

In [13]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question": question})

/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_11110/753188772.py:28: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


'Task decomposition for LLM (Large Language Model) agents involves breaking down large tasks into smaller, manageable subgoals. This is done to enable efficient handling of complex tasks. There are three ways to do task decomposition:\n\n1. LLM with simple prompting: This involves using simple prompts to instruct the LLM to break down the task into smaller subgoals.\n2. Using task-specific instructions: This involves using specific instructions related to the task, such as "Write a story outline" for writing a novel.\n3. With human inputs: This involves using human inputs to guide the task decomposition process.\n\nTask decomposition can be done using techniques such as Chain of Thought (CoT) and Tree of Thoughts (Yao et al. 2023), which transform big tasks into multiple manageable tasks and shed light into an interpretation of the model\'s thinking process.'

## Decomposition

In [14]:
from langchain_core.prompts import ChatPromptTemplate

# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

In [15]:
from langchain_core.output_parsers import StrOutputParser

# Chain
generate_queries_decomposition = (
    prompt_decomposition
    | llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

# Run
question = "What are the main components of an LLM-powered autonomous agent system?"

questions = generate_queries_decomposition.invoke(
    {"question": question}
)

In [16]:
questions

['Here are three search queries related to the main components of an LLM-powered autonomous agent system:',
 '',
 '1. "LLM-based decision-making architectures"',
 '2. "Autonomous agent system design for natural language understanding"',
 '3. "Components of a self-driving autonomous vehicle powered by large language models"',
 '',
 'These sub-questions can be answered in isolation and provide a more detailed understanding of the main components of an LLM-powered autonomous agent system.']

In [17]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question:

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [18]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

def format_qa_pair(question, answer):
    """Format Q and A pair"""

    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

# llm is already initialized using ChatOllama

q_a_pairs = ""
for q in questions:

    rag_chain = (
        {
            "context": itemgetter("question") | retriever,
            "question": itemgetter("question"),
            "q_a_pairs": itemgetter("q_a_pairs")
        }
        | decomposition_prompt
        | llm
        | StrOutputParser()
    )

    answer = rag_chain.invoke({
        "question": q,
        "q_a_pairs": q_a_pairs
    })

    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair

In [19]:
answer

'Based on the provided context and background information, I\'ll attempt to answer the question. However, I must clarify that the question itself is not explicitly stated. Assuming the question is related to the main components of an LLM-powered autonomous agent system, I\'ll provide answers to three potential search queries:\n\n1. **Search Query:** "What is the role of planning in an LLM-powered autonomous agent system?"\n\nAnswer: Planning is a crucial component of an LLM-powered autonomous agent system. It involves breaking down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks. The agent uses planning to decompose tasks, identify potential solutions, and refine its approach based on self-reflection and learning from past mistakes.\n\n2. **Search Query:** "How does memory work in an LLM-powered autonomous agent system?"\n\nAnswer: Memory plays a vital role in LLM-powered autonomous agent systems. It consists of two types: short-term memory, 

In [21]:
# Answer each sub-question individually

from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# RAG prompt from LangSmith
client = Client()
prompt_rag = client.pull_prompt(
    "rlm/rag-prompt",
    dangerously_pull_public_prompt=True
)

def retrieve_and_rag(question, prompt_rag, sub_question_generator_chain):
    """RAG on each sub-question"""

    # Generate sub-questions
    sub_questions = sub_question_generator_chain.invoke(
        {"question": question}
    )

    # Initialize a list to hold RAG chain results
    rag_results = []

    for sub_question in sub_questions:

        # Retrieve documents for each sub-question
        retrieved_docs = retriever.invoke(sub_question)

        # Generate answer for each sub-question
        answer = (
            prompt_rag
            | llm
            | StrOutputParser()
        ).invoke({
            "context": retrieved_docs,
            "question": sub_question
        })

        rag_results.append(answer)

    return rag_results, sub_questions


answers, questions = retrieve_and_rag(
    question,
    prompt_rag,
    generate_queries_decomposition
)

In [22]:
def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""

    formatted_string = ""
    for i, (question, answer) in enumerate(
        zip(questions, answers), start=1
    ):
        formatted_string += (
            f"Question {i}: {question}\n"
            f"Answer {i}: {answer}\n\n"
        )

    return formatted_string.strip()


context = format_qa_pairs(questions, answers)

# Prompt
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({
    "context": context,
    "question": question
})

'Based on the provided Q&A pairs, the main components of an LLM-powered autonomous agent system can be synthesized as follows:\n\n1. Planning: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\n2. Memory: The agent has short-term memory for in-context learning and long-term memory for retaining and recalling information over extended periods, often by leveraging an external vector store and fast retrieval.\n3. Tool use: The agent learns to call external APIs for extra information that is missing from the model weights, including current information, code execution capability, access to proprietary information sources, and more.\n\nThese three components are the main building blocks of an LLM-powered autonomous agent system, as described in the provided Q&A pairs.'

## Step Back

In [1]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]

# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

In [4]:
from langchain_core.output_parsers import StrOutputParser

generate_queries_step_back = prompt | llm | StrOutputParser()

question = "What is task decomposition for LLM agents?"

generate_queries_step_back.invoke({"question": question})

'how do large language models process complex tasks?'

In [9]:
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Response prompt
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""

response_prompt = ChatPromptTemplate.from_template(
    response_prompt_template
)

chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(
            lambda x: x["question"]
        ) | retriever,

        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,

        # Pass on the question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

chain.invoke({"question": question})

'Task decomposition for LLM (Large Language Model) agents is the process of breaking down complex tasks into smaller, manageable subgoals. This is a crucial component of LLM-powered autonomous agent systems, as it enables the agent to efficiently handle complex tasks.\n\nThere are several techniques used for task decomposition, including:\n\n1. **Chain of Thought (CoT)**: This technique involves instructing the LLM to "think step by step" to decompose complex tasks into smaller steps. CoT has become a standard prompting technique for enhancing model performance on complex tasks.\n2. **Tree of Thoughts (ToT)**: This technique extends CoT by exploring multiple reasoning possibilities at each step. ToT decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure.\n3. **Simple prompting**: This involves using simple prompts, such as "Steps for XYZ. 1.", "What are the subgoals for achieving XYZ?", to elicit task decomposition from th

## HyDE

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# HyDE document generation
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""

prompt_hyde = ChatPromptTemplate.from_template(template)

generate_docs_for_retrieval = (
    prompt_hyde | llm | StrOutputParser()
)

# Run
question = "What is task decomposition for LLM agents?"

generate_docs_for_retrieval.invoke({"question": question})

'Here is a scientific paper passage that answers the question:\n\nTask Decomposition for Large Language Model (LLM) Agents: A Framework for Efficient and Effective Task Execution\n\nTask decomposition is a crucial component of Large Language Model (LLM) agent design, enabling the efficient and effective execution of complex tasks. In the context of LLM agents, task decomposition refers to the process of breaking down a high-level task into a set of smaller, more manageable sub-tasks. This decomposition is achieved through a combination of natural language processing (NLP) and machine learning techniques, allowing the LLM agent to generate human-like responses to a wide range of tasks.\n\nThe task decomposition framework for LLM agents involves several key components. Firstly, a task is identified and represented as a high-level concept, such as "book review" or "conversation starter". This high-level task is then decomposed into a set of sub-tasks, such as "extracting relevant informat

In [11]:
# Retrieve
retrieval_chain = generate_docs_for_retrieval | retriever
retrieved_docs = retrieval_chain.invoke({"question": question})
retrieved_docs

[Document(id='f568d180-e953-41dd-99bc-aac3fa6693a5', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Or\n@article{weng2023agent,\n  title   = "LLM-powered Autonomous Agents",\n  author  = "Weng, Lilian",\n  journal = "lilianweng.github.io",\n  year    = "2023",\n  month   = "Jun",\n  url     = "https://lilianweng.github.io/posts/2023-06-23-agent/"\n}\nReferences#\n[1] Wei et al. “Chain of thought prompting elicits reasoning in large language models.” NeurIPS 2022\n[2] Yao et al. “Tree of Thoughts: Dliberate Problem Solving with Large Language Models.” arXiv preprint arXiv:2305.10601 (2023).\n[3] Liu et al. “Chain of Hindsight Aligns Language Models with Feedback\n“ arXiv preprint arXiv:2302.02676 (2023).\n[4] Liu et al. “LLM+P: Empowering Large Language Models with Optimal Planning Proficiency” arXiv preprint arXiv:2304.11477 (2023).'),
 Document(id='92d83641-6e28-4d63-b919-765524f13dc9', metadata={'source': 'https://lilianweng.github.io/posts

In [12]:
# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({
    "context": retrieved_docs,
    "question": question
})

'According to the text, task decomposition for LLM agents involves breaking down large tasks into smaller, manageable subgoals. This can be done in three ways:\n\n1. Using simple prompting, such as "Steps for XYZ. 1." or "What are the subgoals for achieving XYZ?"\n2. Using task-specific instructions, such as "Write a story outline."\n3. With human inputs.\n\nTask decomposition is a key component of LLM-powered autonomous agent systems, allowing agents to efficiently handle complex tasks by dividing them into smaller, more manageable parts.'